# B4 — Clasificador de Contratación Pública

Notebook de clasificación taxonómica para publicaciones del dominio de contratación pública.
Sigue la misma arquitectura incremental que B0-B3.

**Particularidades de B4 frente a bloques anteriores:**
- **Sesgo BOE crítico**: ~59% de la contratación está en BOE (la autonómica y local publica en plataformas de contratación, no en boletines). Documentar en el análisis.
- CONT_CON es un **tipo de contrato**, no una fase: se combina con CONT_LIC/CONT_FOR/CONT_ADJ.
- Fronteras duras: "se adjudica" casi siempre es RRHH (~780 registros de plazas/puestos); "concesión administrativa/demanial" es dominio público (~226 registros, dominio B1).

In [1]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())

True

In [2]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIChatModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

In [3]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")

OK: qwen/qwen3.5-9b listo  |  otros: ['google/gemma-4-e4b', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']


## 1. Schema B4

In [4]:
from clasificador.schema_B4 import (
    ActType, CategoryTypeB4, SubcategoryTypeB4, ClassifierOutputB4
)

In [5]:
ejemplo_valido = ClassifierOutputB4(
    is_relevant=True,
    act_type=ActType.ANUNCIO,
    categories=[CategoryTypeB4.CONT_LIC, CategoryTypeB4.CONT_CON],
    subcategories=[SubcategoryTypeB4.TIPO_SERVICIOS],
    confidence=0.95,
    reasoning="'Anuncio de licitación' → CONT_LIC. 'Objeto: Concesión de servicios para la explotación de cafetería' → CONT_CON + tipo_servicios.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutputB4(
        is_relevant=False, act_type=ActType.ANUNCIO,
        categories=[CategoryTypeB4.CONT_FOR], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "anuncio",
  "categories": [
    "CONT_LIC",
    "CONT_CON"
  ],
  "subcategories": [
    "tipo_servicios"
  ],
  "confidence": 0.95,
  "reasoning": "'Anuncio de licitación' → CONT_LIC. 'Objeto: Concesión de servicios para la explotación de cafetería' → CONT_CON + tipo_servicios."
}

Violación de invariante:
  ValidationError -> Value error, is_relevant=False con categories != []


## 2. Exploración del corpus B4

In [6]:
from clasificador.agent import get_ambito, inferir_act_type

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")

Corpus total: 65,201 registros | Columnas: ['id', 'pdf_link', 'expediente', 'promotor', 'proyecto', 'description', 'tipo', 'clean_id', 'bulletin', 'provincias', 'municipios', 'raw_scraped_timestamp', 'raw_scraped_year_month', 'scraped_timestamp', 'scraped_year_month', 'publication_timestamp', 'publication_type', 'contains_aau', 'proxy_pdf_link', 'ambito', 'rango']


In [7]:
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

keywords_dominio_B4 = [
    "anuncio de licitación", "licitación de", "licitación para",
    "pliego de cláusulas", "pliego de condiciones",
    "formalización del contrato", "formalización de contrato", "formalización contrato",
    "adjudicación del contrato", "adjudicación de contrato",
    "encargo a medio propio", "encomienda de gestión",
    "concesión de servicio", "concesión del servicio", "contrato de concesión",
]
desc_lower = df["description"].str.lower()
mask_b4 = desc_lower.str.contains("|".join(keywords_dominio_B4), na=False)
df_b4 = df[mask_b4].copy()
print(f"Universo B4 estimado: {len(df_b4):,} registros ({len(df_b4)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución por boletín (top 10) - NOTA: sesgo BOE esperado ~59%:")
print(df_b4["bulletin"].value_counts().head(10).to_string())
print(f"\nDistribución N1 en universo B4:")
print(df_b4["act_type_n1"].value_counts().head(8).to_string())

Universo B4 estimado: 3,647 registros (5.59% del corpus)

Distribución por boletín (top 10) - NOTA: sesgo BOE esperado ~59%:
bulletin
boe     2677
bocm     641
boca      75
bopa      66
boib      28
dogv      26
boja      18
borm      17
bon       16
boc       14

Distribución N1 en universo B4:
act_type_n1
anuncio                2814
resolución              762
otros                    32
orden                    11
información_pública       7
corrección_errores        6
acuerdo                   4
edicto                    4


## 3. Ground truth — Muestreo estratificado

Cuotas por grupo (total: 150):

| Grupo | Cuota | Notas |
|-------|-------|-------|
| CONT_FOR | 30 | formalizaciones (la etiqueta dominante del corpus) |
| CONT_LIC | 30 | licitaciones y pliegos |
| CONT_ADJ | 15 | pool pequeño (~60 con keywords estrictas) |
| CONT_ENC | 15 | encargos y encomiendas |
| CONT_CON | 15 | concesiones de servicios/obras (genera multilabel con LIC/FOR) |
| NEG_ADJ_RRHH | 10 | adjudicaciones de plazas/puestos (frontera B5) |
| NEG_DOM_PUB | 10 | concesiones demaniales/administrativas (frontera B1) |
| NEGATIVO | 25 | resto del corpus sin señal B4 |

In [8]:
keywords_B4 = {
    "CONT_LIC": ["anuncio de licitación", "licitación de", "licitación para",
                 "convocatoria de licitación", "pliego de cláusulas",
                 "pliego de condiciones"],
    "CONT_FOR": ["formalización del contrato", "formalización de contrato",
                 "formalización contrato"],
    "CONT_ADJ": ["adjudicación del contrato", "adjudicación de contrato",
                 "adjudicación contrato", "se adjudica el contrato"],
    "CONT_ENC": ["encargo a medio propio", "encargo al medio propio",
                 "encomienda de gestión", "como medio propio"],
    "CONT_CON": ["concesión de servicio", "concesión del servicio",
                 "concesión de obra", "contrato de concesión"],
}

cuotas_B4 = {"CONT_FOR": 30, "CONT_LIC": 30, "CONT_ADJ": 15, "CONT_ENC": 15, "CONT_CON": 15}
N_NEG_ADJ_RRHH = 10
N_NEG_DOM_PUB  = 10
N_NEGATIVOS    = 25

desc_lower = df["description"].str.lower()

def mask_kw(kws):
    return desc_lower.str.contains("|".join(kws), na=False)

# Negativos difíciles
mask_adj_rrhh = (
    (desc_lower.str.contains("se adjudica", na=False) | desc_lower.str.contains("adjudicación de", na=False))
    & desc_lower.str.contains("plaza|puesto|destino", na=False)
    & ~mask_kw(keywords_B4["CONT_ADJ"])
)
mask_dom_pub = (
    desc_lower.str.contains("concesión administrativa|concesión demanial", na=False)
    & ~mask_kw(keywords_B4["CONT_CON"])
)

print(f"{'Label':<14} {'Pool':>8} {'Cuota':>7} {'Estado':>14}")
print("-" * 48)
for label, kws in keywords_B4.items():
    pool = mask_kw(kws).sum()
    cuota = cuotas_B4.get(label, 0)
    estado = "OK" if pool >= cuota else f"REDUCIDA a {min(cuota, pool)}"
    print(f"{label:<14} {pool:>8,} {cuota:>7} {estado:>14}")
print(f"{'NEG_ADJ_RRHH':<14} {mask_adj_rrhh.sum():>8,} {N_NEG_ADJ_RRHH:>7}")
print(f"{'NEG_DOM_PUB':<14} {mask_dom_pub.sum():>8,} {N_NEG_DOM_PUB:>7}")

Label              Pool   Cuota         Estado
------------------------------------------------
CONT_LIC            972      30             OK
CONT_FOR          2,471      30             OK
CONT_ADJ             49      15             OK
CONT_ENC            154      15             OK
CONT_CON             56      15             OK
NEG_ADJ_RRHH        787      10
NEG_DOM_PUB         226      10


In [9]:
sampled_ids = set()
frames = []

def tomar_muestra(pool_df, n, grupo):
    n = min(n, len(pool_df))
    if n == 0:
        print(f"  {grupo:<14} SKIP (pool vacío)")
        return
    sample = pool_df.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<14} pool={len(pool_df):>5,}  sampled={n}")

# 1. Pools pequeños primero
for label in ["CONT_CON", "CONT_ADJ", "CONT_ENC"]:
    mask = mask_kw(keywords_B4[label]) & ~df.index.isin(sampled_ids)
    tomar_muestra(df[mask], cuotas_B4[label], label)

# 2. Negativos difíciles
mask = mask_adj_rrhh & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_NEG_ADJ_RRHH, "NEG_ADJ_RRHH")
mask = mask_dom_pub & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_NEG_DOM_PUB, "NEG_DOM_PUB")

# 3. Pools grandes
for label in ["CONT_LIC", "CONT_FOR"]:
    mask = mask_kw(keywords_B4[label]) & ~df.index.isin(sampled_ids)
    tomar_muestra(df[mask], cuotas_B4[label], label)

# 4. Negativos del resto del corpus
all_kws_b4 = [kw for kws in keywords_B4.values() for kw in kws]
mask_neg = (~desc_lower.str.contains("|".join(all_kws_b4), na=False)
            & ~df.index.isin(sampled_ids))
tomar_muestra(df[mask_neg], N_NEGATIVOS, "NEGATIVO")

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")
print(df_muestreo["grupo_muestreo"].value_counts().to_string())
print(f"\nBOE en la muestra: {(df_muestreo['bulletin'] == 'boe').sum()}/{len(df_muestreo)}")

  CONT_CON       pool=   56  sampled=15
  CONT_ADJ       pool=   49  sampled=15
  CONT_ENC       pool=  154  sampled=15
  NEG_ADJ_RRHH   pool=  787  sampled=10
  NEG_DOM_PUB    pool=  226  sampled=10
  CONT_LIC       pool=  965  sampled=30
  CONT_FOR       pool=2,468  sampled=30
  NEGATIVO       pool=61,525  sampled=25

Total muestreado: 150 registros
grupo_muestreo
CONT_LIC        30
CONT_FOR        30
NEGATIVO        25
CONT_CON        15
CONT_ADJ        15
CONT_ENC        15
NEG_ADJ_RRHH    10
NEG_DOM_PUB     10

BOE en la muestra: 73/150


In [10]:
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul  = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()

Validación del muestreo - 3 ejemplos por grupo:

--- CONT_CON ---
  [BOE] Anuncio de formalización de contratos de: Jefatura de la Sección Económico Administrativa 22 - Base Aérea de Torrejón (Agrupación 
  [BOCYL] RESOLUCIÓN de 7 de enero de 2025, de la Dirección General de Transportes y Logística, por la que se hace pública la transmisión de
  [BOR] Exposición pública del estudio de viabilidad económico-financiera para la concesión de servicio de gestión de las piscinas municip

--- CONT_ADJ ---
  [BOCM] Adjudicación contrato – Anuncio de 6 de febrero de 2025, de adjudicación del contrato titulado “Acuerdo marco para el suministro d
  [BOCM] Convocatoria contrato – Resolución de 4 de marzo de 2025, de la Secretaría General Técnica de la Consejería de Economía, Hacienda 
  [BOCM] Convocatoria contrato – Resolución de 8 de enero de 2025, de la Secretaría General Técnica de la Consejería de Cultura, Turismo y 

--- CONT_ENC ---
  [BON] Encomienda de gestión para recuperar la infraestruc

In [11]:
PATH_MUESTREO = "../data/ground_truth/ground_truth_B4_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")

Guardado: ../data/ground_truth/ground_truth_B4_muestreo.csv  (150 registros)
Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt


## 4. Agente base

In [12]:
from clasificador.schema_B4 import ClassifierOutputB4
from clasificador.prompts_B4 import PROMPT_REGISTRY_B4
from clasificador.agent import build_agent, run_experiment

In [13]:
agent_b4_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

# Casos cualitativos representativos del dominio B4
casos_b4 = [
    ("Anuncio de licitación de: Dirección Provincial del Ministerio de Educación en Melilla. Objeto: Limpieza de 6 Centros Educativos. Expediente: 2024/003.", "boe"),
    ("Anuncio de formalización de contratos de: Jefatura de Asuntos Económicos de la Guardia Civil. Objeto: Suministro de munición para el ejercicio 2025.", "boe"),
    ("Anuncio de licitación de: Subsecretaría de Política Territorial. Objeto: Concesión de servicios para la instalación, explotación y mantenimiento de la cafetería del edificio.", "boe"),
    ("RESOLUCIÓN de 13 de enero de 2025, de la Dirección General de Movilidad, por la que se dispone la publicación de la prórroga del Convenio de encomienda de gestión a TRAGSA para la explotación de estaciones de autobuses.", "doe"),
    ("Resolución de 27 de diciembre de 2024, de la Viceconsejería, por la que se adjudica puesto de trabajo de libre designación convocado por resolución de 30 de octubre.", "bocm"),
]

for desc, bul in casos_b4:
    print(f"\n[{bul.upper()}] {desc[:80]}...")
    # result = await agent_b4_v1.run(f"Boletín: {bul.upper()}\n\nDescripción: {desc}")
    # print(result.output.model_dump_json(indent=2))


[BOE] Anuncio de licitación de: Dirección Provincial del Ministerio de Educación en Me...

[BOE] Anuncio de formalización de contratos de: Jefatura de Asuntos Económicos de la G...

[BOE] Anuncio de licitación de: Subsecretaría de Política Territorial. Objeto: Concesi...

[DOE] RESOLUCIÓN de 13 de enero de 2025, de la Dirección General de Movilidad, por la ...

[BOCM] Resolución de 27 de diciembre de 2024, de la Viceconsejería, por la que se adjud...


## 5. Funciones de evaluación B4

In [14]:
def parse_labels(value) -> set:
    """Convierte cualquier representación de etiquetas a un set de strings."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    s = str(value).strip()
    if s.startswith("["):
        try:
            items = json.loads(s)
            return {str(i).strip('"') for i in items if i}
        except json.JSONDecodeError:
            pass
    return {v.strip().strip('"') for v in s.split(",") if v.strip()}

In [15]:
def compute_metrics_B4(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B4.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    ALL_CATS_B4 = [e.value for e in CategoryTypeB4]
    mlb = MultiLabelBinarizer(classes=ALL_CATS_B4)
    mlb.fit([ALL_CATS_B4])

    gt_labels   = [parse_labels(v) & set(ALL_CATS_B4) for v in df_eval["categories_gt"]]
    pred_labels = [parse_labels(v) & set(ALL_CATS_B4) for v in df_eval["categories_pred"]]
    Y    = mlb.transform(gt_labels)
    Yhat = mlb.transform(pred_labels)

    is_rel_gt   = df_eval["is_relevant_gt"].astype(bool)
    is_rel_pred = df_eval["is_relevant_pred"].astype(bool)

    exact = pd.Series([set(g) == set(p) for g, p in zip(gt_labels, pred_labels)])
    rel   = is_rel_gt

    metrics = {
        "is_rel_accuracy":  round(accuracy_score(is_rel_gt, is_rel_pred), 4),
        "is_rel_precision": round(precision_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_recall":    round(recall_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_f1":        round(f1_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "micro_f1":         round(f1_score(Y, Yhat, average="micro", zero_division=0), 4),
        "macro_f1":         round(f1_score(Y, Yhat, average="macro", zero_division=0), 4),
        "hamming_loss":     round(hamming_loss(Y, Yhat), 4),
        "jaccard_samples":  round(jaccard_score(Y, Yhat, average="samples", zero_division=0), 4),
        "subset_accuracy":  round(accuracy_score(Y, Yhat), 4),
        "exact": exact,
        "rel":   rel,
    }

    if verbose:
        title = f"-- {label} --" if label else "-- Métricas B4 --"
        print(f"\n{title}\n")
        tp = int((is_rel_gt & is_rel_pred).sum())
        fp = int((~is_rel_gt & is_rel_pred).sum())
        fn = int((is_rel_gt & ~is_rel_pred).sum())
        tn = int((~is_rel_gt & ~is_rel_pred).sum())
        print(f"is_relevant  Acc={metrics['is_rel_accuracy']:.3f}  P={metrics['is_rel_precision']:.3f}  "
              f"R={metrics['is_rel_recall']:.3f}  F1={metrics['is_rel_f1']:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {metrics['micro_f1']:.3f}")
        print(f"  Macro F1:      {metrics['macro_f1']:.3f}")
        print(f"  Hamming Loss:  {metrics['hamming_loss']:.4f}")
        print(f"  Jaccard:       {metrics['jaccard_samples']:.3f}")
        print(f"  Subset Acc:    {metrics['subset_accuracy']:.3f}\n")
        f1s = f1_score(Y, Yhat, average=None, zero_division=0)
        sups = Y.sum(axis=0)
        print(f"  {'Label':<12} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print(f"  {'-'*37}")
        for i, cat in enumerate(ALL_CATS_B4):
            prec = precision_score(Y[:, i], Yhat[:, i], zero_division=0)
            rec  = recall_score(Y[:, i], Yhat[:, i], zero_division=0)
            print(f"  {cat:<12} {prec:>6.3f} {rec:>6.3f} {f1s[i]:>6.3f} {int(sups[i]):>5}")
        print(f"  {'-'*37}")
        print(f"  {'Macro':<12} {'':>6} {'':>6} {metrics['macro_f1']:>6.3f}\n")

        cards = [len(g) for g in gt_labels]
        print(f"  Subset Acc por cardinalidad:")
        for card in [0, 1, 2]:
            idx = [i for i, c in enumerate(cards) if c == card]
            if idx:
                acc = accuracy_score(Y[idx], Yhat[idx])
                print(f"    card={card} ({'no relevante' if card == 0 else str(card)}) : {acc:.3f}  ({int(acc*len(idx))}/{len(idx)})")
        idx3 = [i for i, c in enumerate(cards) if c >= 3]
        if idx3:
            acc3 = accuracy_score(Y[idx3], Yhat[idx3])
            print(f"    card>=3               : {acc3:.3f}  ({int(acc3*len(idx3))}/{len(idx3)})")

        conf = df_eval.get("confidence", pd.Series(dtype=float))
        if conf.notna().any():
            print(f"\n  Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return metrics

In [16]:
def print_errors_B4(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    lbl = f" · {label}" if label else ""
    print(f"\n-- Errores N2 en relevantes{lbl} --")
    print(f"Total: {len(errores)}\n")
    for _, row in errores.iterrows():
        gt   = sorted(parse_labels(row["categories_gt"]))
        pred = sorted(parse_labels(row["categories_pred"]))
        falta = sorted(set(gt) - set(pred))
        sobra = sorted(set(pred) - set(gt))
        desc  = str(row["description"])[:90].replace("\n", " ")
        razon = str(row.get("reasoning", "")).replace("\n", " ")[:120]
        print(f"ID {row['id']} | GT={gt} | PRED={pred}")
        print(f"  Falta: {falta} | Sobra: {sobra}")
        print(f"  {desc}...")
        print(f"  Razonamiento: {razon}")
        print()

---

##  6. Experimento 1 - Baseline zero-shot

**Problema**: No existe un clasificador para el dominio de contratación pública. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: Establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistemáticos. Atención especial a: (1) frontera adjudicación de contrato vs adjudicación de plaza (RRHH), (2) frontera concesión contractual vs concesión demanial (B1), (3) multilabel CONT_CON + fase.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V1 · zero-shot · sin contexto N1.

**Resultados**: is_rel F1=0.935 (P=1.000, R=0.878, **12 FN**) · Micro F1=0.881 · **Macro F1=0.840** · Subset Acc=0.840.

| Label | P | R | F1 | Sup |
|-------|------|------|------|-----|
| CONT_LIC | 0.972 | 0.972 | 0.972 | 36 |
| CONT_FOR | 0.969 | 1.000 | 0.984 | 31 |
| CONT_ADJ | 1.000 | 0.900 | 0.947 | 10 |
| CONT_ENC | 1.000 | 0.600 | 0.750 | 15 |
| CONT_CON | 0.474 | 0.643 | **0.545** | 14 |

**Análisis de errores (24 en relevantes)**: las dos reglas frontera de V1 sobre-corrigieron y hay una confusión conceptual nueva:
1. **CONT_CON espuria (10 errores, P=0.474)**: confunde "contrato DE servicios" con "CONCESIÓN de servicios": añade CONT_CON a licitaciones y formalizaciones ordinarias de vigilancia, limpieza, soporte (ids 66, 74, 77, 91, 99, 102, 104, 105, 112, 121).
2. **Regla demanial sobre-aplicada (5 FN: ids 2, 3, 4, 7, 9)**: marca como demaniales concesiones de servicios LCSP reales (cafetería de la Fuerza Terrestre, piscinas municipales, abastecimiento de Alcantarilla, aparcamiento de Oviedo).
3. **Regla de convenios sobre-aplicada (6 FN: ids 33, 34, 38, 40, 42, 44)**: descarta los convenios BOPA que formalizan encomiendas de gestión de ayuda a domicilio (R de CONT_ENC=0.600).
4. Menores: "se formaliza la encomienda" → CONT_FOR espuria (id 41), FOR+LIC espuria (id 109), adjudicación con mención de subvenciones → [] (id 23).

Los negativos están perfectos (52/52): las fronteras RRHH, DOP y demanial-pura funcionan. El problema es que el modelo no distingue el lado relevante de cada frontera. V2 reescribe las 3 reglas con señales de ambos lados.

In [17]:
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B4_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"].notna()].copy()

df_b4_exp1 = await run_experiment(
    agent_b4_v1, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp1_baseline_qwen9b.csv",
    desc="B4 Exp1 - Baseline V1",
)
df_b4_exp1.head(3)

  Reanudando: 150/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s,error
0,0,boe,Anuncio de formalización de contratos de: Jefa...,CONT_CON,True,"CONT_FOR, CONT_CON",tipo_servicios,Formalizacion de concesion de servicios cafeteria,True,anuncio,"[""CONT_FOR"", ""CONT_CON""]","[""tipo_servicios""]",0.95,"El texto contiene explícitamente ""Anuncio de f...",120.410,NaN
1,1,bocyl,"RESOLUCIÓN de 7 de enero de 2025, de la Direcc...",CONT_CON,True,CONT_CON,tipo_servicios,Transmision de titularidad de contrato de conc...,True,resolución,"[""CONT_CON""]","[""tipo_servicios""]",0.98,"El texto menciona explícitamente ""transmisión ...",57.966,NaN
2,2,bor,Exposición pública del estudio de viabilidad e...,CONT_CON,True,CONT_CON,tipo_servicios,Estudio de viabilidad previo a concesion de se...,False,información_pública,[],[],0.95,"La publicación es una ""Exposición pública del ...",58.667,NaN


In [18]:
df_b4_exp1 = pd.read_csv("../results/b4_exp1_baseline_qwen9b.csv")
df_eval_b4_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_1 = compute_metrics_B4(df_eval_b4_1, "Experimento B4-1 - Baseline")
print_errors_B4(df_eval_b4_1, m_b4_1["exact"], m_b4_1["rel"], label="B4 Experimento 1 - Baseline")


-- Experimento B4-1 - Baseline --

is_relevant  Acc=0.920  P=1.000  R=0.878  F1=0.935
             TP=86  FP=0  FN=12  TN=52

N2 multilabel:
  Micro F1:      0.881
  Macro F1:      0.840
  Hamming Loss:  0.0333
  Jaccard:       0.533
  Subset Acc:    0.840

  Label             P      R     F1   Sup
  -------------------------------------
  CONT_LIC      0.972  0.972  0.972    36
  CONT_FOR      0.969  1.000  0.984    31
  CONT_ADJ      1.000  0.900  0.947    10
  CONT_ENC      1.000  0.600  0.750    15
  CONT_CON      0.474  0.643  0.545    14
  -------------------------------------
  Macro                       0.840

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 1.000  (52/52)
    card=1 (1) : 0.744  (67/90)
    card=2 (2) : 0.875  (7/8)

  Confianza: media=0.970  min=0.850  max=1.000

-- Errores N2 en relevantes · B4 Experimento 1 - Baseline --
Total: 24

ID 2 | GT=['CONT_CON'] | PRED=[]
  Falta: ['CONT_CON'] | Sobra: []
  Exposición pública del estudio de viabilidad e

---

##  7. Experimento 2 - Prompt v2 (correcciones post-baseline)

**Problema**: El baseline (Macro F1=0.840) tiene precisión 1.000 en negativos pero las reglas frontera sobre-corrigieron hacia dentro: CONT_CON espuria en contratos ordinarios de servicios (P=0.474), 5 concesiones LCSP reales descartadas como demaniales y 6 encomiendas descartadas como convenios interadministrativos (R de CONT_ENC=0.600).

**Objetivo**: Verificar si V2 corrige los 3 patrones con reglas bidireccionales: (1) "contrato DE servicios ≠ CONCESIÓN de servicios" (CONT_CON exige la palabra concesión en el objeto), (2) frontera demanial/LCSP con señales de ambos lados (ocupación de dominio público vs concesión de servicios de cafetería/piscinas/agua/transporte), (3) el convenio que formaliza una encomienda ES CONT_ENC, y los actos preparatorios de una concesión (viabilidad, estructura de costes) son CONT_CON.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V2 (4.066 chars) · zero-shot · sin contexto N1.

**Resultados**: pendiente.

> Nota: para ejecutar tras actualizar prompts_B4.py hay que reiniciar el kernel (el import de PROMPT_REGISTRY_B4 no se recarga solo).

In [19]:
agent_b4_v2 = build_agent(
    model, "v2",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_exp2 = await run_experiment(
    agent_b4_v2, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp2_promptv2_qwen9b.csv",
    desc="B4 Exp2 - Prompt v2",
)
df_b4_exp2.head(3)

  Reanudando: 148/150 registros ya clasificados


B4 Exp2 - Prompt v2: 100%|██████████| 2/2 [04:09<00:00, 124.55s/it]

Tiempo: 249.1s total  |  124.55s/item  |  2 items


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s,error
0,0,boe,Anuncio de formalización de contratos de: Jefa...,CONT_CON,True,"CONT_FOR, CONT_CON",tipo_servicios,Formalizacion de concesion de servicios cafeteria,True,anuncio,"[""CONT_FOR"", ""CONT_CON""]","[""tipo_servicios""]",0.98,"El texto es un ""Anuncio de formalización de co...",133.115,NaN
1,1,bocyl,"RESOLUCIÓN de 7 de enero de 2025, de la Direcc...",CONT_CON,True,CONT_CON,tipo_servicios,Transmision de titularidad de contrato de conc...,True,resolución,"[""CONT_CON""]",[],0.95,"La publicación es una resolución sobre la ""tra...",66.447,NaN
2,2,bor,Exposición pública del estudio de viabilidad e...,CONT_CON,True,CONT_CON,tipo_servicios,Estudio de viabilidad previo a concesion de se...,True,información_pública,"[""CONT_CON""]",[],0.95,Es un acto de información pública relativo a u...,65.514,NaN


In [20]:
df_b4_exp2 = pd.read_csv("../results/b4_exp2_promptv2_qwen9b.csv")
df_eval_b4_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_2 = compute_metrics_B4(df_eval_b4_2, "Experimento B4-2 - Prompt v2")
print_errors_B4(df_eval_b4_2, m_b4_2["exact"], m_b4_2["rel"], label="B4 Experimento 2 - Prompt v2")


-- Experimento B4-2 - Prompt v2 --

is_relevant  Acc=0.960  P=0.979  R=0.959  F1=0.969
             TP=94  FP=2  FN=4  TN=50

N2 multilabel:
  Micro F1:      0.971
  Macro F1:      0.958
  Hamming Loss:  0.0080
  Jaccard:       0.627
  Subset Acc:    0.960

  Label             P      R     F1   Sup
  -------------------------------------
  CONT_LIC      1.000  1.000  1.000    36
  CONT_FOR      1.000  0.935  0.967    31
  CONT_ADJ      1.000  0.800  0.889    10
  CONT_ENC      0.938  1.000  0.968    15
  CONT_CON      0.933  1.000  0.966    14
  -------------------------------------
  Macro                       0.958

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.962  (50/52)
    card=1 (1) : 0.956  (86/90)
    card=2 (2) : 1.000  (8/8)

  Confianza: media=0.969  min=0.850  max=1.000

-- Errores N2 en relevantes · B4 Experimento 2 - Prompt v2 --
Total: 4

ID 23 | GT=['CONT_ADJ'] | PRED=[]
  Falta: ['CONT_ADJ'] | Sobra: []
  Resolución de 16 de enero de 2025, de la Secr

---

##  8. Experimento 3 - Prompt v3 (few-shot)

**Problema**: (rellenar tras el análisis de errores del §7)

**Objetivo**: Verificar si ejemplos few-shot quirúrgicos sobre los errores residuales mejoran el resultado de V2.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V3 · few-shot · sin contexto N1.

**Resultados**: pendiente.

> ⚠️ **Bloqueado**: SYSTEM_PROMPT_B4_V3 se escribirá tras analizar los errores del Experimento 2.

In [21]:
agent_b4_v3 = build_agent(
    model, "v3",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_exp3 = await run_experiment(
    agent_b4_v3, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp3_promptv3_qwen9b.csv",
    desc="B4 Exp3 - Prompt v3",
)
df_b4_exp3.head(3)

B4 Exp3 - Prompt v3:   3%|▎         | 5/150 [07:02<3:43:32, 92.50s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B4 Exp3 - Prompt v3: 100%|██████████| 150/150 [2:17:46<00:00, 55.11s/it] 

Tiempo: 8266.7s total  |  55.11s/item  |  150 items


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s,error
0,0,boe,Anuncio de formalización de contratos de: Jefa...,CONT_CON,True,"CONT_FOR, CONT_CON",tipo_servicios,Formalizacion de concesion de servicios cafeteria,True,anuncio,"[""CONT_FOR"", ""CONT_CON""]","[""tipo_servicios""]",0.98,"El texto indica claramente ""Anuncio de formali...",80.824,NaN
1,1,bocyl,"RESOLUCIÓN de 7 de enero de 2025, de la Direcc...",CONT_CON,True,CONT_CON,tipo_servicios,Transmision de titularidad de contrato de conc...,True,resolución,"[""CONT_CON"", ""CONT_ADJ""]","[""tipo_servicios""]",0.95,"El texto describe la ""transmisión de la titula...",73.054,NaN
2,2,bor,Exposición pública del estudio de viabilidad e...,CONT_CON,True,CONT_CON,tipo_servicios,Estudio de viabilidad previo a concesion de se...,True,información_pública,"[""CONT_CON""]",[],0.95,"El texto describe una ""Exposición pública del ...",69.419,NaN


In [22]:
df_b4_exp3 = pd.read_csv("../results/b4_exp3_promptv3_qwen9b.csv")
df_eval_b4_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_3 = compute_metrics_B4(df_eval_b4_3, "Experimento B4-3 - Prompt v3")
print_errors_B4(df_eval_b4_3, m_b4_3["exact"], m_b4_3["rel"], label="B4 Experimento 3 - Prompt v3")


-- Experimento B4-3 - Prompt v3 --

is_relevant  Acc=0.987  P=0.990  R=0.990  F1=0.990
             TP=97  FP=1  FN=1  TN=51

N2 multilabel:
  Micro F1:      0.986
  Macro F1:      0.976
  Hamming Loss:  0.0040
  Jaccard:       0.643
  Subset Acc:    0.980

  Label             P      R     F1   Sup
  -------------------------------------
  CONT_LIC      1.000  1.000  1.000    36
  CONT_FOR      1.000  1.000  1.000    31
  CONT_ADJ      0.909  1.000  0.952    10
  CONT_ENC      1.000  1.000  1.000    15
  CONT_CON      0.929  0.929  0.929    14
  -------------------------------------
  Macro                       0.976

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.981  (51/52)
    card=1 (1) : 0.978  (88/90)
    card=2 (2) : 1.000  (8/8)

  Confianza: media=0.967  min=0.000  max=1.000

-- Errores N2 en relevantes · B4 Experimento 3 - Prompt v3 --
Total: 2

ID 1 | GT=['CONT_CON'] | PRED=['CONT_ADJ', 'CONT_CON']
  Falta: [] | Sobra: ['CONT_ADJ']
  RESOLUCIÓN de 7 de enero

---

##  9. Comparativa de modelos - Gemma 4B

**Problema**: Todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos más pequeños y rápidos.

**Objetivo**: Comprobar si Gemma 4 4B con el mejor prompt compacto alcanza un rendimiento comparable al 9B.

**Enfoque**: Gemma 4 4B · mejor prompt que quepa en su ventana de contexto · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [23]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio")
)
agent_b4_gemma = build_agent(
    model_gemma, "v2",  # ajustar a la mejor versión compacta disponible
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_gemma = await run_experiment(
    agent_b4_gemma, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp_gemma4b.csv",
    desc="B4 Gemma4B",
)
df_b4_gemma.head(3)

B4 Gemma4B:  37%|███▋      | 56/150 [40:54<1:16:38, 48.92s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B4 Gemma4B:  38%|███▊      | 57/150 [41:49<1:18:43, 50.79s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B4 Gemma4B:  39%|███▊      | 58/150 [43:10<1:31:38, 59.76s/it]

...

B4 Gemma4B:  43%|████▎     | 64/150 [48:09<1:11:20, 49.78s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B4 Gemma4B: 100%|██████████| 150/150 [1:48:12<00:00, 43.29s/it]

Tiempo: 6492.9s total  |  43.29s/item  |  150 items


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s,error
0,0,boe,Anuncio de formalización de contratos de: Jefa...,CONT_CON,True,"CONT_FOR, CONT_CON",tipo_servicios,Formalizacion de concesion de servicios cafeteria,True,anuncio,"[""CONT_FOR"", ""CONT_CON""]","[""tipo_servicios""]",0.95,"Se menciona explícitamente ""Anuncio de formali...",96.569,NaN
1,1,bocyl,"RESOLUCIÓN de 7 de enero de 2025, de la Direcc...",CONT_CON,True,CONT_CON,tipo_servicios,Transmision de titularidad de contrato de conc...,True,resolución,"[""CONT_CON""]","[""tipo_servicios""]",0.98,"El acto trata sobre la ""transmisión de la titu...",37.401,NaN
2,2,bor,Exposición pública del estudio de viabilidad e...,CONT_CON,True,CONT_CON,tipo_servicios,Estudio de viabilidad previo a concesion de se...,True,información_pública,"[""CONT_CON""]","[""tipo_servicios""]",0.95,"El texto se refiere a la ""Exposición pública d...",41.049,NaN


In [24]:
df_b4_gemma = pd.read_csv("../results/b4_exp_gemma4b.csv")
df_eval_b4_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_gemma = compute_metrics_B4(df_eval_b4_gemma, "B4 Gemma 4B")
print_errors_B4(df_eval_b4_gemma, m_b4_gemma["exact"], m_b4_gemma["rel"], label="B4 Gemma 4B")


-- B4 Gemma 4B --

is_relevant  Acc=0.973  P=0.961  R=1.000  F1=0.980
             TP=98  FP=4  FN=0  TN=48

N2 multilabel:
  Micro F1:      0.968
  Macro F1:      0.962
  Hamming Loss:  0.0093
  Jaccard:       0.647
  Subset Acc:    0.960

  Label             P      R     F1   Sup
  -------------------------------------
  CONT_LIC      0.947  1.000  0.973    36
  CONT_FOR      0.969  1.000  0.984    31
  CONT_ADJ      0.909  1.000  0.952    10
  CONT_ENC      0.938  1.000  0.968    15
  CONT_CON      0.875  1.000  0.933    14
  -------------------------------------
  Macro                       0.962

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.923  (48/52)
    card=1 (1) : 0.978  (88/90)
    card=2 (2) : 1.000  (8/8)

  Confianza: media=0.960  min=0.200  max=1.000

-- Errores N2 en relevantes · B4 Gemma 4B --
Total: 2

ID 10 | GT=['CONT_CON'] | PRED=['CONT_CON', 'CONT_LIC']
  Falta: [] | Sobra: ['CONT_LIC']
  Resolución de 24 de febrero de 2025, de la Dirección Gere

## 10. Tabla resumen - Comparativa de experimentos B4

In [25]:
experimentos_cfg_B4 = [
    ("B4 Exp1 - Baseline V1", "Qwen 3.5 9B", "Zero-shot", "../results/b4_exp1_baseline_qwen9b.csv"),
    ("B4 Exp2 - Prompt V2",   "Qwen 3.5 9B", "Zero-shot", "../results/b4_exp2_promptv2_qwen9b.csv"),
    ("B4 Exp3 - Prompt V3",   "Qwen 3.5 9B", "Few-shot",  "../results/b4_exp3_promptv3_qwen9b.csv"),
    ("B4 Exp4 - Gemma 4B",    "Gemma 4 4B",  "Zero-shot", "../results/b4_exp_gemma4b.csv"),
]

rows_b4 = []
for nombre, modelo, config, path in experimentos_cfg_B4:
    if not Path(path).exists():
        print(f"  [SKIP] {nombre}: aún sin resultados ({path})")
        continue
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B4(df_e, verbose=False)
    rows_b4.append({
        "Experimento":     nombre,
        "Modelo":          modelo,
        "Config":          config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b4 = pd.DataFrame(rows_b4)
if not df_summary_b4.empty:
    df_display_b4 = df_summary_b4.copy()
    df_display_b4.columns = [
        "Experimento", "Modelo", "Config",
        "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
        "s/item", "Total (s)",
    ]
    display(df_display_b4.set_index("Experimento"))

,Modelo,Config,is_rel F1,Micro F1,Macro F1,Hamming,Jaccard,Subset Acc,s/item,Total (s)
Experimento,,,,,,,,,,
B4 Exp1 - Baseline V1,Qwen 3.5 9B,Zero-shot,0.9348,0.8815,0.8398,0.0333,0.5333,0.84,63.88,9583.0
B4 Exp2 - Prompt V2,Qwen 3.5 9B,Zero-shot,0.9691,0.9714,0.9578,0.0080,0.6267,0.96,62.71,9407.0
B4 Exp3 - Prompt V3,Qwen 3.5 9B,Few-shot,0.9898,0.9859,0.9762,0.0040,0.6433,0.98,55.11,8266.0
B4 Exp4 - Gemma 4B,Gemma 4 4B,Zero-shot,0.9800,0.9680,0.9621,0.0093,0.6467,0.96,43.28,6492.0
